# PI4 — EDA de Guaratinguetá

Notebook da etapa **E01**. Ele só prossegue se a base materializada pela execução humana do Colab passar no gate de reconciliação de P04.

A análise aqui é descritiva: volume de escolas/matrículas, infraestrutura, rede, zona e permanência das escolas entre 2023 e 2025.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_NAME = 'UNIVESP — PI4 — Infraestrutura Escolar — Guaratinguetá'
candidates = [Path('/content/drive/MyDrive') / PROJECT_NAME, Path('/content/drive/My Drive') / PROJECT_NAME]
PROJECT_ROOT = next((p for p in candidates if p.exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Adicione um atalho da pasta compartilhada do PI4 ao Meu Drive.')

BASE_ANALITICA = PROJECT_ROOT / '01_Dados' / '2_tratamentos_dados' / 'base_analitica'
print(PROJECT_ROOT)


In [ ]:
!rm -rf /content/pi4_repo
!git clone -q --depth 1 https://github.com/felipecsr/univesp-projeto-integrador-4.git /content/pi4_repo
sys.path.insert(0, '/content/pi4_repo')

from src.validate_analytic_base import validate_execution
from src.eda import (
    INFRA_DIMENSIONS,
    latest_execution_dir,
    load_materialized_panel,
    guaratingueta_overview,
    infrastructure_by_year,
    network_summary,
    localization_summary,
    school_presence,
)

REPO_COMMIT = subprocess.check_output(['git','-C','/content/pi4_repo','rev-parse','HEAD'], text=True).strip()
RUN_DIR = latest_execution_dir(BASE_ANALITICA)
print('Commit:', REPO_COMMIT)
print('Base usada:', RUN_DIR)


## Gate P04

Se houver qualquer `FAIL`, o notebook interrompe antes da EDA.


In [ ]:
reconciliacao = validate_execution(RUN_DIR)
display(reconciliacao)
assert not (reconciliacao['status'] == 'FAIL').any(), 'P04 falhou. Corrija a base antes da EDA.'
print('GATE P04: PASS')


In [ ]:
panel = load_materialized_panel(RUN_DIR)
overview = guaratingueta_overview(panel)
infra = infrastructure_by_year(panel)
rede = network_summary(panel)
zona = localization_summary(panel)
presenca = school_presence(panel)

display(overview)


## Infraestrutura 2023–2025

Percentuais abaixo representam **escolas**, não alunos. Nulos permanecem fora do denominador válido e são registrados separadamente.


In [ ]:
infra_tabela = infra.pivot(index=['dimensao','indicador'], columns='ano', values='pct_escolas_com_item').reset_index()
display(infra_tabela)

for dimensao in infra['dimensao'].drop_duplicates():
    grupo = infra[infra['dimensao'] == dimensao]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for indicador, serie in grupo.groupby('indicador'):
        ax.plot(serie['ano'], serie['pct_escolas_com_item'] * 100, marker='o', label=indicador)
    ax.set_title(f'Guaratinguetá — {dimensao}')
    ax.set_ylabel('% de escolas com o item')
    ax.set_xlabel('Ano')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=8, loc='best')
    ax.grid(axis='y', alpha=0.25)
    plt.show()


## Rede e localização


In [ ]:
display(rede)
display(zona)

fig, ax = plt.subplots(figsize=(8, 4.5))
for rede_nome, serie in rede.groupby('rede'):
    ax.plot(serie['ano'], serie['escolas'], marker='o', label=rede_nome)
ax.set_title('Guaratinguetá — escolas ativas por rede')
ax.set_ylabel('Escolas')
ax.set_xlabel('Ano')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.show()


## Permanência das escolas

Este quadro ajuda a distinguir mudança de infraestrutura de mudança no próprio conjunto de escolas ativas.


In [ ]:
presenca_resumo = presenca.groupby('n_anos_presente').size().rename('escolas').reset_index()
display(presenca_resumo)
display(presenca.sort_values(['n_anos_presente','NO_ENTIDADE']).head(30))


## Materialização da EDA

A execução cria uma pasta própria em `2_tratamentos_dados/eda` e não altera a base validada de P03/P04.


In [ ]:
EDA_ROOT = PROJECT_ROOT / '01_Dados' / '2_tratamentos_dados' / 'eda'
EDA_RUN = EDA_ROOT / f'guaratingueta_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
EDA_RUN.mkdir(parents=True, exist_ok=True)

overview.to_csv(EDA_RUN / 'resumo_anual_guaratingueta.csv', index=False, encoding='utf-8')
infra.to_csv(EDA_RUN / 'infraestrutura_guaratingueta.csv', index=False, encoding='utf-8')
rede.to_csv(EDA_RUN / 'rede_guaratingueta.csv', index=False, encoding='utf-8')
zona.to_csv(EDA_RUN / 'localizacao_guaratingueta.csv', index=False, encoding='utf-8')
presenca.to_csv(EDA_RUN / 'presenca_escolas_guaratingueta.csv', index=False, encoding='utf-8')
reconciliacao.to_csv(EDA_RUN / 'gate_p04.csv', index=False, encoding='utf-8')

manifest = pd.DataFrame([{
    'executado_em': datetime.now().isoformat(timespec='seconds'),
    'repo_commit': REPO_COMMIT,
    'base_origem': str(RUN_DIR),
    'linhas_municipio': int((panel['CO_MUNICIPIO'].astype(str) == '3518404').sum()),
}])
manifest.to_csv(EDA_RUN / 'manifesto_eda_guaratingueta.csv', index=False, encoding='utf-8')
display(manifest)
print('EDA gravada em:', EDA_RUN)


## Evidências manuais de E01

Preserve o notebook com outputs e registre: **(1)** gate P04 PASS; **(2)** resumo anual; **(3)** uma visualização de infraestrutura; **(4)** quadro de permanência das escolas. Depois, deixe o card `E01M` em `Revisão`.
